# การทำนายผลลัพธ์แบบทดสอบตามรูปแบบตารางอาจารย์
### โครงสร้างตาราง: `Path | Image Name | Extension | Label | Result`
- **Label:** ช่องเฉลย (Ground Truth) หากมีข้อมูล จะคำนวณ Accuracy (%) ให้อัตโนมัติ
- **Result:** ช่องคำตอบที่โมเดลทำนาย (Predicted Class) โค้ดจะเติมค่าลงในช่องนี้
- **รองรับ:** ทั้งไฟล์ `.csv` และ `.xlsx` / `.xls` พร้อมไฟล์ภาพทุกประเภท (.gif, .png, .jpg, .bmp ฯลฯ)


In [1]:
import os
import sys
import glob
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import resnet18
import torchvision.transforms.functional as TF

plt.rcParams['font.family'] = 'Tahoma'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 1. โครงสร้าง ResNet-18 (Transfer Learning 72 คลาส)
def build_model(num_classes=72):
    model = resnet18(weights=None)
    model.fc = nn.Sequential(
        nn.Dropout(0.2),
        nn.Linear(model.fc.in_features, num_classes)
    )
    return model

# 2. แปลงรหัสคลาส (TIS-620) เป็นตัวอักษรภาษาไทย
def get_thai_char(c):
    try:
        return bytes([int(c)]).decode('tis-620')
    except Exception:
        return str(c)

# 3. ฟังก์ชันโหลดรูปภาพรองรับ GIF, PNG, JPG, BMP และจัดการพื้นหลังโปร่งใส (Transparency)
def load_image(img_path):
    img = Image.open(img_path)
    if hasattr(img, 'is_animated') and img.is_animated:
        img.seek(0)
    if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
        img = img.convert('RGBA')
        bg = Image.new('RGBA', img.size, (255, 255, 255, 255))
        bg.paste(img, mask=img.split()[3])
        return bg.convert('RGB')
    return img.convert('RGB')

# 4. โหลดโมเดล model.pt
checkpoint = torch.load('model.pt', map_location=device)
sd = checkpoint.get('model_state_dict', checkpoint)

classes = [str(c) for c in checkpoint.get('classes', [str(i) for i in range(161, 250)])]
idx_to_class = {i: str(c) for i, c in enumerate(classes)}
classes_str = set(classes)

model = build_model(num_classes=len(classes))
model.load_state_dict({k.replace('backbone.', ''): v for k, v in sd.items()})
model = model.to(device)
model.eval()

# 5. Data Transform (64x64)
img_size = checkpoint.get('img_size', 64)
transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
print(f'โหลดโมเดลสำเร็จ! (Image Size: {img_size}x{img_size}, Classes: {len(classes)})')


Using device: cuda:0
โหลดโมเดลสำเร็จ! (Image Size: 64x64, Classes: 72)


## 1. ฟังก์ชันทำนายจากตารางของอาจารย์ และส่งออก out.csv
ฟังก์ชันนี้จะ:
1. อ่านไฟล์ตารางโจทย์ (`test.csv` หรือ `test.xlsx`)
2. นำ `Path` + `Image Name` + `Extension` มาประกอบเป็น Path ของรูปภาพ
3. ป้อนภาพเข้าโมเดล ResNet-18 (เปิดใช้ TTA เพื่อความแม่นยำสูงสุด)
4. หยอดผลทำนายลงในช่อง **`Result`**
5. ตรวจสอบช่อง **`Label`** หากมีเฉลย จะคำนวณและแสดงค่า Accuracy (%) ออกมาทันที
6. บันทึกตารางทั้งหมดออกเป็นไฟล์ `out.csv`


In [2]:
def load_table_file(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    if ext in ('.xlsx', '.xls'):
        try:
            return pd.read_excel(file_path)
        except ImportError:
            print('[INFO] กำลังติดตั้ง openpyxl...')
            import subprocess
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openpyxl'])
            return pd.read_excel(file_path)
    else:
        for enc in ['utf-8', 'utf-8-sig', 'tis-620', 'cp874', 'latin1']:
            try:
                temp_df = pd.read_csv(file_path, encoding=enc)
                first_col = str(temp_df.columns[0])
                if ';' in first_col:
                    return pd.read_csv(file_path, sep=';', encoding=enc)
                elif '\t' in first_col:
                    return pd.read_csv(file_path, sep='\t', encoding=enc)
                return temp_df
            except Exception:
                continue
        return pd.read_csv(file_path)

def predict_teacher_format(table_path='test.csv', img_root='.', output_csv='out.csv', use_tta=True):
    # ค้นหาไฟล์อัตโนมัติหากไม่พบ
    if not os.path.exists(table_path):
        base, _ = os.path.splitext(table_path)
        candidates = [table_path, f'{base}.xlsx', f'{base}.xls', f'{base}.csv',
                      'test/test.csv', 'test/test.xlsx', 'test.csv', 'test.xlsx']
        for c in candidates:
            if os.path.exists(c):
                table_path = c
                break
        else:
            print(f'[ERROR] ไม่พบไฟล์ตาราง: {table_path}')
            return None

    print(f'[INFO] กำลังโหลดตารางข้อมูลจาก: {table_path}')
    df = load_table_file(table_path)

    # ค้นหาชื่อคอลัมน์แบบยืดหยุ่น (ไม่สนใจตัวพิมพ์เล็ก/ใหญ่ หรือช่องว่างหัวท้าย)
    col_map = {str(c).strip().lower(): c for c in df.columns}
    
    path_col = col_map.get('path', None)
    name_col = col_map.get('image name', col_map.get('imagename', col_map.get('name', None)))
    ext_col = col_map.get('extension', col_map.get('ext', None))
    label_col = col_map.get('label', None)
    result_col = col_map.get('result', None)

    if not path_col or not name_col:
        print(f"[ERROR] ไม่พบคอลัมน์ 'Path' หรือ 'Image Name' ในตาราง (คอลัมน์ที่มี: {list(df.columns)})")
        return None

    if result_col is None:
        result_col = 'Result'
        df[result_col] = None

    predictions = []
    has_labels = False
    correct_count = 0
    total_labeled = 0

    print(f'[INFO] กำลังประมวลผลทั้งหมด {len(df):,} รายการ...')
    with torch.no_grad():
        for idx, row in df.iterrows():
            sub_path = str(row[path_col]).strip() if pd.notna(row[path_col]) else ''
            img_name = str(row[name_col]).strip() if pd.notna(row[name_col]) else ''
            extension = str(row[ext_col]).strip() if ext_col and pd.notna(row[ext_col]) else ''
            
            # ประกอบร่างชื่อไฟล์
            full_filename = f'{img_name}{extension}' if not img_name.endswith(extension) else img_name

            # ค้นหาไฟล์ใน root paths ต่างๆ
            search_paths = [
                os.path.normpath(os.path.join(img_root, sub_path, full_filename)),
                os.path.normpath(os.path.join(sub_path, full_filename)),
                os.path.normpath(os.path.join('test', sub_path, full_filename)),
                os.path.normpath(os.path.join(img_root, full_filename))
            ]

            img_path = None
            for sp in search_paths:
                if os.path.exists(sp) and os.path.isfile(sp):
                    img_path = sp
                    break

            if img_path and os.path.exists(img_path):
                img = load_image(img_path)
                if use_tta:
                    tensors = [
                        transform(img.rotate(angle, fillcolor=(255, 255, 255), resample=Image.BILINEAR)).unsqueeze(0).to(device)
                        for angle in [-5, 0, 5]
                    ]
                    probs = torch.softmax(model(torch.cat(tensors, dim=0)), dim=1).mean(dim=0, keepdim=True)
                else:
                    tensor = transform(img).unsqueeze(0).to(device)
                    probs = torch.softmax(model(tensor), dim=1)

                pred_idx = probs.argmax(dim=1).item()
                pred_code = str(idx_to_class[pred_idx])
                label_val = int(pred_code) if pred_code.isdigit() else pred_code
                predictions.append(label_val)

                # ตรวจสอบว่าช่อง Label มีเฉลยหรือไม่
                if label_col and pd.notna(row[label_col]) and str(row[label_col]).strip() != '':
                    true_val = str(row[label_col]).strip()
                    if true_val.endswith('.0'):
                        true_val = true_val[:-2]
                    total_labeled += 1
                    has_labels = True
                    if str(label_val) == true_val:
                        correct_count += 1
            else:
                predictions.append(None)

    # หยอดคำตอบลงในคอลัมน์ Result
    df[result_col] = predictions

    # บันทึกไฟล์ผลลัพธ์เป็น out.csv
    df.to_csv(output_csv, index=False)

    # สรุปผลการประเมิน
    if has_labels and total_labeled > 0:
        acc = (correct_count / total_labeled) * 100
        print(f'[EVAL] Accuracy: {acc:.2f}% (ถูกต้อง {correct_count:,}/{total_labeled:,} ภาพ)')
    else:
        print('[INFO] ไม่พบข้อมูลในช่อง "Label" (ส่งคำตอบครบ รออาจารย์ตรวจ)')

    print(f"[OK] บันทึกผลการทำนายลง '{output_csv}' เรียบร้อยครบถ้วน ({len(df):,} แถว)")
    return df.head(10)


## 2. คำสั่งประมวลผล (Run Prediction)
สั่งรันเพื่ออ่านไฟล์ของอาจารย์และส่งออก `out.csv`:


In [3]:
# เรียกใช้งานฟังก์ชัน
# กำหนด table_path เป็นชื่อไฟล์ที่อาจารย์ให้มา (เช่น test.csv หรือ test.xlsx)
predict_teacher_format(table_path='test.csv', img_root='.', output_csv='out.csv', use_tta=True)


[INFO] กำลังโหลดตารางข้อมูลจาก: test/test.csv
[ERROR] ไม่พบคอลัมน์ 'Path' หรือ 'Image Name' ในตาราง (คอลัมน์ที่มี: ['id'])


## 3. สุ่มตรวจภาพและคำตอบด้วยสายตา (Visual Spot Check)
ใช้ฟังก์ชันนี้สุ่มตรวจดูรูปภาพเทียบกับผลลัพธ์ที่โมเดลทายลงในช่อง `Result` พร้อมตัวอักษรไทย:


In [4]:
def spot_check(csv_file='out.csv', img_root='.', n=8):
    if not os.path.exists(csv_file):
        print(f'[ERROR] ไม่พบไฟล์ {csv_file}')
        return
        
    df = pd.read_csv(csv_file)
    if 'Result' not in df.columns or df['Result'].dropna().empty:
        print('[ERROR] ยังไม่มีผลลัพธ์ในคอลัมน์ Result')
        return

    valid_rows = df[df['Result'].notna()]
    sample_df = valid_rows.sample(min(n, len(valid_rows)))

    cols = 4
    rows = int(np.ceil(len(sample_df) / cols))
    plt.figure(figsize=(cols * 3.5, rows * 3.5))

    for i, (_, row) in enumerate(sample_df.iterrows()):
        sub_path = str(row['Path']).strip() if 'Path' in row and pd.notna(row['Path']) else ''
        img_name = str(row['Image Name']).strip()
        ext = str(row['Extension']).strip() if 'Extension' in row and pd.notna(row['Extension']) else ''
        full_filename = f'{img_name}{ext}' if not img_name.endswith(ext) else img_name
        
        search_paths = [
            os.path.normpath(os.path.join(img_root, sub_path, full_filename)),
            os.path.normpath(os.path.join(sub_path, full_filename)),
            os.path.normpath(os.path.join('test', sub_path, full_filename))
        ]
        
        img_path = None
        for sp in search_paths:
            if os.path.exists(sp):
                img_path = sp
                break
                
        if img_path:
            img = load_image(img_path)
            res_code = str(row['Result']).replace('.0', '')
            res_char = get_thai_char(res_code)
            
            lbl_info = f"\n(เฉลย: {row['Label']})" if 'Label' in row and pd.notna(row['Label']) else ''
            
            plt.subplot(rows, cols, i + 1)
            plt.imshow(img)
            plt.axis('off')
            plt.title(f"{img_name}\nทาย: {res_char} (รหัส {res_code}){lbl_info}", fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.show()

# เรียกสุ่มตรวจ 8 ภาพตัวอย่าง (หากมี out.csv แล้ว)
if os.path.exists('out.csv'):
    spot_check('out.csv', img_root='.', n=8)


[ERROR] ยังไม่มีผลลัพธ์ในคอลัมน์ Result
